In [1]:
from dataclasses import dataclass, field
from functools import partial
from typing import Optional
import logging
import json
import math
import sys
import os

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from peft import (
    get_peft_config,
    get_peft_model,
    get_peft_model_state_dict,
    set_peft_model_state_dict,
    TaskType,
    PeftType,
    PrefixTuningConfig,
    PromptEncoderConfig,
    LoraConfig
)

from trl import SFTTrainer, SFTConfig

# data
from preprocess import (
    CustomDataset,
    torchdataset_generator,
    DataCollatorForSupervisedDataset,
    # collate_fn,
    Collate,
    lambda_unsqeeze,
    PreprocessData
)

from datasets import Dataset, Features, Value, Sequence

from datasets import load_dataset
from transformers import HfArgumentParser, AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from transformers.trainer_callback import TrainerCallback # peft의 경우 Trainer로 잘 저장이 안되는 이슈가 있음. // 최신버전에서는 없어졋난봄.
from tqdm import tqdm
import argparse



/home/rainism/anaconda3/envs/LLM/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
class Config:
    model_name_or_path = 'beomi/Llama-3-Open-Ko-8B'
    train_data_path = '/data1/kaggle/Korean_DCS_2024/data/일상대화요약_train.json'
    valid_data_path = '/data1/kaggle/Korean_DCS_2024/data/일상대화요약_test.json'
    max_length = 4096

config = Config

tokenizer = AutoTokenizer.from_pretrained(config.model_name_or_path)
tokenizer.pad_token = tokenizer.eos_token

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [21]:
train_data_object = PreprocessData(
    config.train_data_path
)
train_df = train_data_object.make_columns()

valid_data_object = PreprocessData(
    config.valid_data_path
)
valid_df = valid_data_object.make_columns()

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)


In [22]:
# IGNORE_INDEX = -100


# def instruction_prompt(prompt_function, subject_keyword, speaker1, speaker2):
#     prompt = prompt_function(subject_keyword, speaker1, speaker2)
#     return prompt

# def merge_chat_with_instruction_prompt(chat:str, instruction:str)->str:
#     end_prompt = instruction
#     result = chat+'\n\n' + end_prompt
#     return result

# def message_prompt(system_prompt, chat_prompt):
#     return [{'role' : 'system', 'content' : system_prompt},
#             {'role' : 'user', 'content' : chat_prompt}]

# def wrapping_function(tokenizer, conversation, system_prompt_function, chat_prompt_function, subject_keyword, speaker1, speaker2, output, config):

#     instruct_prompt = instruction_prompt(chat_prompt_function,  subject_keyword, speaker1, speaker2)
#     merge_chat_with_instruct_prompt = merge_chat_with_instruction_prompt(conversation, instruct_prompt)
#     message = message_prompt(system_prompt_function, merge_chat_with_instruct_prompt)
    
#     source = tokenizer.apply_chat_template(
#         message,
#         add_generation_prompt=True,
#         return_tensors='pt'
#     )

#     target = tokenizer(output,
#             return_attention_mask=False,
#             add_special_tokens=False,
#             return_tensors="pt",
#             truncation=True, 
#             max_length=config.max_length)
    
#     input_ids = torch.concat((source.squeeze(0), target['input_ids'].squeeze(0)))
#     labels = torch.concat((torch.LongTensor([IGNORE_INDEX] * source.shape[1]), target["input_ids"].squeeze(0)))

#     return {'input_ids':input_ids, 'labels':labels}


# def tokenize_function(x, tokenizer, config):
#     return [wrapping_function(tokenizer, chat, prompt_template.SYSTEM_PROMPT_1, prompt_template.USER_PROMPT_1, subject_keyword, speaker1, speaker2, output, config) 
#     for speaker1, speaker2, subject_keyword, chat, output in zip(x['speaker1'], x['speaker2'], x['subject_keyword'], x['chat_data'], x['output'])]

# train_dataset.map(lambda x: tokenize_function(x, tokenizer, config))



Map:   0%|          | 0/506 [00:00<?, ? examples/s]


TypeError: Provided `function` which is applied to all elements of table returns a variable of type <class 'list'>. Make sure provided `function` returns a variable of type `dict` (or a pyarrow table) to update the dataset or `None` if you are only interested in side effects.

In [17]:
def merge_chat_with_prompt(
        chat:str, 
        prompt:str
    )->str:
    end_prompt = prompt
    out = chat + '\n\n' + end_prompt
    return out

aa = train_dataset[0]
s1 = aa['speaker1']
s2 = aa['speaker2']
subject = aa['subject_keyword']
conv = aa['chat_data']


from prompt import prompt_template

prompt = prompt_template.USER_PROMPT_1({'subject_keyword':subject, 'user1':s1, 'user2':s2})
chat = merge_chat_with_prompt(conv, prompt)


message = [
    {'role' : 'system', 'content' : prompt_template.SYSTEM_PROMPT_1},
    {'role' : 'user', 'content' : chat}
]


import pprint

pprint.pprint(tokenizer.apply_chat_template(message, tokenize=False))


TypeError: <lambda>() missing 2 required positional arguments: 'user1' and 'user2'

In [12]:
aa

{'id': 'nikluge-2024-일상 대화 요약-train-000001',
 'input': {'conversation': [{'speaker': 'SD2000001',
    'utterance': '저는 여행 다니는 것을 굉장히 좋아하는데요. 그래가지고 스페인이나 뭐 영국 유럽 아니면 국내에서도 뭐 강릉이나 전주 같은 데를 많이 다녔는데'},
   {'speaker': 'SD2000001', 'utterance': '혹시 여행 다니는 거 좋아하시나요?'},
   {'speaker': 'SD2000002',
    'utterance': '저 여행 다니는 거 되게 좋아해서 대학교 내내 여행을 엄청 많이 다녔었는데요.'},
   {'speaker': 'SD2000002',
    'utterance': '제가 고등학교 때는 여행에 대해 흥미가 없었는데 그게 좀 아버지가 짠대로 패키지처럼 여행을 다녀서 그런 것 같아요.'},
   {'speaker': 'SD2000002',
    'utterance': '그래서 대학교 간 이후로는 해외여행을 되게 많이 갔었는데 그중에서 제일 기 좋았던 거는 스페인이랑 포르투갈이었거든요.'},
   {'speaker': 'SD2000002',
    'utterance': '어~ 혹시 포르투갈이나 스페인 유럽 쪽 다녀오신 적 있으신가요?'},
   {'speaker': 'SD2000001', 'utterance': '어~ 네. 저도 우연히 스페인과 포르투갈을 다녀왔었었습니다.'},
   {'speaker': 'SD2000001',
    'utterance': '어~ 저는 스페인 중에서도 마드리드에 근교에 있었던 톨레도라는 지역이 굉장히 좋았는데요. 그 톨레도에서 특히 기억에 남았던 거는 거기에 대성당이 있는데 그 성당이 엄청 화려하더라고요. 그래서 거기를 꾸며논 거를 보면은 금을 엄청 많이 사용해가지고 되게 빤짝빤짝하고 좀 성당은 보통 좀 소박하다라는 인식이 있었는데 아~ 이렇게 화려한 성당도 있구나라는 거를 새롭게 알게